In [2]:
import pandas as pd

df = pd.read_csv("optimization/output/allocation_x.csv")

print(
    df[df["allocated_quantity"] > 1e-6]
    .groupby("supply_level")["allocated_quantity"]
    .sum()
)


supply_level
state    31.195214
Name: allocated_quantity, dtype: float64


In [3]:
print(df["allocated_quantity"].max())


0.4164878493640273


In [4]:
u = pd.read_csv("optimization/output/unmet_demand_u.csv")
print(u["unmet_demand"].sum())


0.0


In [7]:
import pandas as pd

s = pd.read_csv("scenarios/generated/state_resource_stock_C2.csv")
d = pd.read_csv("scenarios/generated/scenario_A_demand_shock.csv")

peak = (
    d.groupby(["resource_id", "time"])["demand"]
     .sum()
     .groupby("resource_id")
     .max()
)

stock = s.groupby("resource_id")["quantity"].sum()

print(pd.concat([peak, stock], axis=1).rename(
    columns={"demand": "peak_demand", "quantity": "state_stock"}
))


             peak_demand  state_stock
resource_id                          
R1             62.563228    50.050584
R10            68.819551    55.055641
R11            87.588520    70.070819
R2             62.563228    50.050584
R3             75.075874    60.060700
R4             81.332197    65.065755
R5             93.844842    75.075874
R6            100.101165    80.080933
R7             75.075874    60.060699
R8             50.050583    40.040467
R9             56.306905    45.045523


In [17]:
import pandas as pd
alloc = pd.read_csv("optimization/output/allocation_x.csv")


print(alloc.groupby("supply_level")["allocated_quantity"].sum())
print("Unmet:", unmet["unmet_demand"].sum())

print(
    alloc.groupby(["resource_id", "supply_level"])
    ["allocated_quantity"].sum()
)


supply_level
district    7.966649
state       4.608712
Name: allocated_quantity, dtype: float64
Unmet: 0.0
resource_id  supply_level
R1           district        0.682881
             state           0.284454
R10          district        0.656802
             state           0.407267
R11          district        0.439638
             state           0.914631
R2           district        0.598162
             state           0.369174
R3           district        0.725301
             state           0.435502
R4           district        0.962856
             state           0.294680
R5           district        0.843878
             state           0.607126
R6           district        1.105532
             state           0.442205
R7           district        0.742588
             state           0.418215
R8           district        0.517713
             state           0.256156
R9           district        0.691300
             state           0.179302
Name: allocated_quantity, dtype

In [13]:
import pandas as pd

alloc = pd.read_csv(
    r"C:\Users\LATHEEF\Desktop\disaster_management\phase4\optimization\output\allocation_x.csv"
)


demand = pd.read_csv(
    r"C:\Users\LATHEEF\Desktop\disaster_management\phase3\output\district_resource_demand.csv"
)

print("Allocation rows:", len(alloc))

Allocation rows: 149


In [14]:
alloc_sum = (
    alloc
    .groupby(["resource_id", "district_code", "time"])["allocated_quantity"]
    .sum()
    .reset_index()
)

merged = alloc_sum.merge(
    demand,
    on=["resource_id", "district_code", "time"],
    how="left"
)

merged["gap"] = merged["demand"] - merged["allocated_quantity"]

print(merged["gap"].describe())


count    871.000000
mean    -100.339581
std       49.239867
min     -134.999063
25%     -134.985312
50%     -134.972049
75%      -59.967680
max       -0.935733
Name: gap, dtype: float64


In [15]:
print(
    alloc.groupby("supply_level")["allocated_quantity"].sum()
)


supply_level
district    6271.0
national    3537.4
state        518.7
Name: allocated_quantity, dtype: float64


In [16]:
import pandas as pd

unmet = pd.read_csv(
    r"C:\Users\LATHEEF\Desktop\disaster_management\phase4\optimization\output\unmet_demand_u.csv"
)

print(unmet.head())
print("Unique unmet values:", unmet["unmet_demand"].unique())
print("Rows:", len(unmet))


  resource_id  district_code  time  unmet_demand
0         R10            100     0           0.0
1         R10            100     1           0.0
2         R10            101     0           0.0
3         R10            101     1           0.0
4         R10            102     0           0.0
Unique unmet values: [  0.   123.    50.   125.   115.25  77.3   73.55 131.   132.    57.
 134.   117.85 129.   125.05 114.9  135.    65.4   49.15  60.75  94.
  98.   115.    48.85  83.  ]
Rows: 14080


In [18]:
import pandas as pd

# Path to your existing scenario summary
CSV_PATH = "optimization/summary/scenario_summary.csv"

# Load the table
df = pd.read_csv(CSV_PATH)

# Scenario descriptions (authoritative catalogue)
scenario_descriptions = {
    "S1": "Zero-Demand Baseline: Validates system inactivity when demand is zero, ensuring no phantom allocations or forced escalation.",
    
    "S3": "Single-District Shock: Simulates a localized disaster affecting one district, validating correct locality handling and escalation only when necessary.",
    
    "S4": "Multi-District Intra-State Surge: Models simultaneous demand across multiple districts within a state, validating state-level pooling and coordination.",
    
    "S5": "State Collapse to National Escalation: Forces exhaustion of state resources, validating necessary and correct escalation to national supply.",
    
    "S6": "Population Skew Fairness: Tests ethical fairness under unequal population distribution, ensuring smaller districts are not starved.",
    
    "S12": "Total Failure Transparency: Simulates demand exceeding all available resources, validating explicit unmet demand reporting under unavoidable failure."
}

# Add description column
df["scenario_description"] = df["scenario_id"].map(scenario_descriptions)

# Reorder columns (description next to scenario_id)
cols = ["scenario_id", "scenario_description"] + [c for c in df.columns if c not in ["scenario_id", "scenario_description"]]
df = df[cols]

# Save back to CSV
df.to_csv(CSV_PATH, index=False)

df


,scenario_id,scenario_description,total_demand,total_allocated,total_unmet,unmet_pct,district_supply_used,state_supply_used,national_supply_used,national_used_flag,escalation_level,worst_hit_district,max_unmet_value,fairness_cv,status
0,S12,Total Failure Transparency: Simulates demand e...,3619.381067,10327.1,2872.9,0.793754,6271.0,518.7,3537.4,1,national,605.0,720.35,0.000000,ESCALATED
1,S1,Zero-Demand Baseline: Validates system inactiv...,3619.381067,0.0,0.0,0.000000,0.0,0.0,0.0,0,district,1.0,0.00,0.000000,OK
2,S3,Single-District Shock: Simulates a localized d...,3619.381067,990.0,0.0,0.000000,405.0,45.0,540.0,1,national,1.0,0.00,0.534191,OK
3,S4,Multi-District Intra-State Surge: Models simul...,3619.381067,2640.0,0.0,0.000000,1721.0,463.0,456.0,1,national,1.0,0.00,0.000000,OK
4,S5,State Collapse to National Escalation: Forces ...,3619.381067,7040.0,0.0,0.000000,3742.0,265.9,3032.1,1,national,1.0,0.00,0.000000,OK
5,S6,Population Skew Fairness: Tests ethical fairne...,3619.381067,4356.0,0.0,0.000000,1964.0,1111.0,1281.0,1,national,1.0,0.00,0.000000,OK


In [19]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

p=Path("optimization/summary/scenario_summary.csv")
o=Path("visualizations")
o.mkdir(parents=True,exist_ok=True)

d=pd.read_csv(p)

x=d.scenario_id
a=d.total_allocated
u=d.total_unmet

plt.figure(figsize=(10,6))
plt.bar(x,a)
plt.bar(x,u,bottom=a)
plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Scenario Outcomes: Allocation vs Unmet Demand")
plt.tight_layout()
plt.savefig(o/"scenario_outcomes.png")
plt.close()

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

p=Path("optimization/summary/scenario_summary.csv")
o=Path("visualizations")
o.mkdir(parents=True,exist_ok=True)

d=pd.read_csv(p)

x=d.scenario_id
d1=d.district_supply_used
d2=d.state_supply_used
d3=d.national_supply_used

plt.figure(figsize=(10,6))
plt.bar(x,d1)
plt.bar(x,d2,bottom=d1)
plt.bar(x,d3,bottom=d1+d2)
plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Escalation Profile by Supply Level")
plt.tight_layout()
plt.savefig(o/"escalation_profile.png")
plt.close()


In [21]:
SCENARIO_MAP = {
    "S1":  {"order": 1, "alias": "S1", "label": "S1: Zero Demand"},
    "S3":  {"order": 2, "alias": "S2", "label": "S2: Single-District Shock"},
    "S4":  {"order": 3, "alias": "S3", "label": "S3: Multi-District Surge"},
    "S5":  {"order": 4, "alias": "S4", "label": "S4: State Collapse"},
    "S6":  {"order": 5, "alias": "S5", "label": "S5: Population Skew"},
    "S12": {"order": 6, "alias": "S6", "label": "S6: Total Failure"}
}


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SCENARIO_MAP = {
    "S1":  {"order": 1, "alias": "S1", "label": "S1: Zero Demand"},
    "S3":  {"order": 2, "alias": "S2", "label": "S2: Single-District Shock"},
    "S4":  {"order": 3, "alias": "S3", "label": "S3: Multi-District Surge"},
    "S5":  {"order": 4, "alias": "S4", "label": "S4: State Collapse"},
    "S6":  {"order": 5, "alias": "S5", "label": "S5: Population Skew"},
    "S12": {"order": 6, "alias": "S6", "label": "S6: Total Failure"}
}

p = Path("optimization/summary/scenario_summary.csv")
o = Path("visualizations")
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

df["order"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["order"])
df["label"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["label"])

df = df.sort_values("order")

x = df.label
allocated = df.total_allocated
unmet = df.total_unmet

plt.figure(figsize=(11,6))
plt.bar(x, allocated, label="Allocated", color="#4C72B0")
plt.bar(x, unmet, bottom=allocated, label="Unmet Demand", color="#DD8452")

plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Scenario Outcomes: Allocation vs Unmet Demand")
plt.legend()
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(o / "scenario_outcomes.png")
plt.close()


In [24]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SCENARIO_MAP = {
    "S1":  {"order": 1, "label": "S1: Zero Demand"},
    "S3":  {"order": 2, "label": "S2: Single-District Shock"},
    "S4":  {"order": 3, "label": "S3: Multi-District Surge"},
    "S5":  {"order": 4, "label": "S4: State Collapse"},
    "S6":  {"order": 5, "label": "S5: Population Skew"},
    "S12": {"order": 6, "label": "S6: Total Failure"}
}

p = Path("optimization/summary/scenario_summary.csv")
o = Path("visualizations")
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

df["order"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["order"])
df["label"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["label"])
df = df.sort_values("order")

x = df.label
d = df.district_supply_used
s = df.state_supply_used
n = df.national_supply_used

plt.figure(figsize=(11,6))
plt.bar(x, d, label="District Supply", color="#55A868")
plt.bar(x, s, bottom=d, label="State Supply", color="#E17C05")
plt.bar(x, n, bottom=d+s, label="National Supply", color="#8172B2")

plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Escalation Profile by Supply Level")
plt.legend()
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(o / "escalation_profile.png")
plt.close()


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SCENARIO_MAP = {
    "S1":  {"order": 1, "label": "S1: Zero Demand"},
    "S3":  {"order": 2, "label": "S2: Single-District Shock"},
    "S4":  {"order": 3, "label": "S3: Multi-District Surge"},
    "S5":  {"order": 4, "label": "S4: State Collapse"},
    "S6":  {"order": 5, "label": "S5: Population Skew"},
    "S12": {"order": 6, "label": "S6: Total Failure"}
}

p = Path("optimization/summary/scenario_summary.csv")
o = Path("visualizations")
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

df["order"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["order"])
df["label"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["label"])
df = df.sort_values("order")

x = df.label
d = df.district_supply_used
s = df.state_supply_used
n = df.national_supply_used

plt.figure(figsize=(11,6))
plt.bar(x, d, label="District Supply", color="#55A868")
plt.bar(x, s, bottom=d, label="State Supply", color="#E17C05")
plt.bar(x, n, bottom=d+s, label="National Supply", color="#8172B2")

plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Escalation Profile by Supply Level")
plt.legend()
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(o / "escalation_profile.png")
plt.close()


In [27]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

p = Path("optimization/output/unmet_demand_u.csv")
o = Path("visualizations")
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

g = df.groupby("district_code")["unmet_demand"].sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(g.index.astype(str), g.values, color="#DD8452")

plt.xlabel("District")
plt.ylabel("Unmet Demand")
plt.title("S6: Total Failure — Unmet Demand by District")
plt.xticks(rotation=90)

plt.tight_layout()
plt.savefig(o / "s6_unmet_by_district.png")
plt.close()


In [29]:
from pathlib import Path

# Find project root by walking upwards until phase4 exists
cwd = Path.cwd().resolve()
PROJECT_ROOT = None

for p in [cwd] + list(cwd.parents):
    if (p / "phase4").exists() and (p / "data").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Project root not found. Run notebook inside the project directory.")

PROJECT_ROOT


WindowsPath('C:/Users/LATHEEF/Desktop/disaster_management')

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

alloc_path = PROJECT_ROOT / "phase4" / "optimization" / "output" / "allocation_x.csv"
pop_path = PROJECT_ROOT / "data" / "processed" / "new_data" / "clean_district_population.csv"
out_dir = PROJECT_ROOT / "phase4" / "visualizations"
out_dir.mkdir(parents=True, exist_ok=True)

alloc = pd.read_csv(alloc_path)
pop = pd.read_csv(pop_path)[["district_code", "population"]]

g = alloc.groupby("district_code")["allocated_quantity"].sum().reset_index()
m = g.merge(pop, on="district_code", how="left")

# Remove invalid population
m = m[m["population"] > 0]

# Compute per-capita
m["per_capita"] = m["allocated_quantity"] / m["population"]

# Remove inf / NaN
pc = m["per_capita"].replace([np.inf, -np.inf], np.nan).dropna()

# Plot
plt.figure(figsize=(8, 6))
plt.hist(pc, bins=20, color="#4C72B0", edgecolor="black")

plt.xlabel("Per-Capita Allocation")
plt.ylabel("Number of Districts")
plt.title("Fairness Distribution Across Districts (Population > 0)")

plt.tight_layout()
plt.savefig(out_dir / "fairness_distribution.png")
plt.close()


In [38]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

p = PROJECT_ROOT / "phase4" / "optimization" / "output" / "unmet_demand_u.csv"
o = PROJECT_ROOT / "phase4" / "visualizations"
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

# Aggregate and filter
g = (
    df.groupby("district_code")["unmet_demand"]
    .sum()
    .reset_index()
)

g = g[g.unmet_demand > 0]
g = g.sort_values("unmet_demand", ascending=False).head(10)

plt.figure(figsize=(10,6))
plt.bar(g.district_code.astype(str), g.unmet_demand, color="#DD8452")

plt.xlabel("District")
plt.ylabel("Unmet Demand")
plt.title("S6: Total Failure — Top Districts by Unmet Demand")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(o / "s6_unmet_by_district.png")
plt.close()


In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# --- Load data ---
p = PROJECT_ROOT / "phase4" / "optimization" / "summary" / "scenario_summary.csv"
o = PROJECT_ROOT / "phase4" / "visualizations"
o.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(p)

# --- Scenario ordering & labels ---
SCENARIO_MAP = {
    "S1":  {"order": 1, "label": "S1: Zero Demand"},
    "S3":  {"order": 2, "label": "S2: Single-District Shock"},
    "S4":  {"order": 3, "label": "S3: Multi-District Surge"},
    "S5":  {"order": 4, "label": "S4: State Collapse"},
    "S6":  {"order": 5, "label": "S5: Population Skew"},
    "S12": {"order": 6, "label": "S6: Total Failure"},
}

df["order"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["order"])
df["label"] = df.scenario_id.map(lambda x: SCENARIO_MAP[x]["label"])
df = df.sort_values("order")

# --- Data ---
allocated = df.total_allocated.values
unmet = df.total_unmet.values
labels = df.label.values

x = np.arange(len(labels))
w = 0.35

# --- Plot ---
plt.figure(figsize=(11, 6))

plt.bar(x - w/2, allocated, width=w, label="Allocated", color="#4C72B0")
plt.bar(x + w/2, unmet, width=w, label="Unmet Demand", color="#DD8452")

plt.xticks(x, labels, rotation=30, ha="right")
plt.xlabel("Scenario")
plt.ylabel("Resource Units")
plt.title("Scenario Outcomes: Allocation vs Unmet Demand")
plt.legend()

plt.tight_layout()
plt.savefig(o / "scenario_outcomes_allocation_vs_unmet.png")
plt.close()


In [37]:
df = pd.read_csv(p)

print("Total unmet demand:", df["unmet_demand"].sum())
print("Max unmet demand:", df["unmet_demand"].max())

df[df["unmet_demand"] > 0].head(10)


Total unmet demand: 0.0
Max unmet demand: 0.0


,resource_id,district_code,time,unmet_demand
